In [0]:
import requests
from datetime import datetime, timezone
from urllib.parse import urlencode
import time
import json
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)


catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
api_key = dbutils.secrets.get(
    catalog = catalog,
    schema= schema,
    key="owm_api_token"
)

assert api_key, "Secret is empty"
print("Secret read successfully")


In [0]:
url = "https://api.openweathermap.org/data/2.5/forecast"
city , request_url, http_status, raw_payload,ingestion_timestamp = [], [], [], [],[]
cities = [
    "London,uk",
    "New York,NY,us",
    "Tokyo,jp",
    "Sydney,au",
    "Paris,fr"
]
for location in cities:

    params = {
        "q": location,
        "appid": api_key,
    }

    safe_url = url + "?" + urlencode({
        "q": location    })

    response = requests.get(url, params=params, timeout=30)

    retry_count = 0
    max_retries = 3

    while (
        response.status_code == 429
        or 500 <= response.status_code <= 599
    ) and retry_count < max_retries:

        wait_seconds = 2 ** retry_count 

        print(
            f"{location}: HTTP {response.status_code}. "
            f"Retrying in {wait_seconds} seconds..."
        )

        time.sleep(wait_seconds)

        response = requests.get(url, params=params, timeout=30)
        retry_count += 1

    city.append(location)
    request_url.append(safe_url)
    http_status.append(response.status_code)

    try:
        raw_payload.append(json.dumps(response.json()))
    except ValueError:
        raw_payload.append(response.text)

    ingestion_timestamp.append(datetime.now(timezone.utc))

bronze_record = [
    {
        "city": city_value,
        "request_url": url_value,
        "http_status": status_value,
        "raw_payload": payload_value,
        "ingestion_timestamp": timestamp_value
    }
    for city_value, url_value, status_value, payload_value, timestamp_value
    in zip(
        city,
        request_url,
        http_status,
        raw_payload,
        ingestion_timestamp
    )
]

In [0]:

bronze_df = spark.createDataFrame(bronze_record)
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog}.{schema}.weather_bronze")

In [0]:
%sql
truncate table dev_demo_catalog.weather_analytics.weather_silver

### Silver Table if not exist 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_demo_catalog.weather_analytics.weather_silver(
city string, 
forecast_unix_timestamp bigint,
forecast_timestamp timestamp,
 temperature double,
 feels_like double,
 humidity int,
 pressure int,
 weather_main string,
 weather_description string,
 wind_speed double, 
 wind_direction int, 
 ingestion_timestamp timestamp,
 weather_hash BIGINT)
USING DELTA;

### Merge 

In [0]:
%sql
-- Make timestamp conversions display in UTC.
SET TIME ZONE 'UTC';

MERGE INTO dev_demo_catalog.weather_analytics.weather_silver AS target
USING (
    WITH forecasts AS (
        SELECT
            city,
            ingestion_timestamp,
            explode(
                from_json(
                    raw_payload:list,
                    'ARRAY<STRUCT<
                        dt: BIGINT,
                        dt_txt: STRING,
                        main: STRUCT<
                            temp: DOUBLE,
                            feels_like: DOUBLE,
                            humidity: INT,
                            pressure: INT
                        >,
                        weather: ARRAY<STRUCT<
                            main: STRING,
                            description: STRING
                        >>,
                        wind: STRUCT<
                            speed: DOUBLE,
                            deg: INT
                        >
                    >>'
                )
            ) AS forecast
        FROM dev_demo_catalog.weather_analytics.weather_bronze
        WHERE http_status = 200
    ),

    parsed_weather AS (
        SELECT
            city,
            forecast.dt AS forecast_unix_timestamp,
            timestamp_seconds(forecast.dt) AS forecast_timestamp,

            -- Standard API units are Kelvin.
            round(forecast.main.temp - 273.15,2) AS temperature,
            round(forecast.main.feels_like - 273.15,2) AS feels_like,

            forecast.main.humidity AS humidity,
            forecast.main.pressure AS pressure,
            forecast.weather[0].main AS weather_main,
            forecast.weather[0].description AS weather_description,
            forecast.wind.speed AS wind_speed,
            forecast.wind.deg AS wind_direction,
            ingestion_timestamp
        FROM forecasts
        WHERE city IS NOT NULL
          AND forecast.dt IS NOT NULL
    ),

    forecasts_window AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY city, forecast_unix_timestamp
                ORDER BY ingestion_timestamp DESC
            ) AS ingestion_number
        FROM parsed_weather
    )

    SELECT
        *,
        xxhash64(
            temperature,
            feels_like,
            humidity,
            pressure,
            weather_main,
            weather_description,
            wind_speed,
            wind_direction
        ) AS weather_hash
    FROM forecasts_window
    WHERE ingestion_number = 1
) AS source

ON  target.city = source.city
AND target.forecast_unix_timestamp = source.forecast_unix_timestamp

WHEN MATCHED
    AND NOT (target.weather_hash <=> source.weather_hash)
    AND source.ingestion_timestamp >= target.ingestion_timestamp
THEN UPDATE SET *

WHEN NOT MATCHED
THEN INSERT *;

In [0]:
%sql
SELECT * FROM dev_demo_catalog.weather_analytics.weather_silver

In [0]:
%sql
CREATE OR REPLACE VIEW
    dev_demo_catalog.weather_analytics.weather_gold_summary AS

WITH daily_weather AS (
    SELECT
        city,
        CAST(forecast_timestamp AS DATE) AS forecast_date,
        temperature,
        humidity,
        wind_speed,
        weather_main
    FROM dev_demo_catalog.weather_analytics.weather_silver
),

daily_metrics AS (
    SELECT
        city,
        forecast_date,
        AVG(temperature) AS avg_temperature_celsius,
        MIN(temperature) AS min_temperature_celsius,
        MAX(temperature) AS max_temperature_celsius,
        AVG(humidity) AS avg_humidity,
        MAX(wind_speed) AS max_wind_speed_mps,
        COUNT(*) AS data_point_count
    FROM daily_weather
    GROUP BY city, forecast_date
),

weather_frequency AS (
    SELECT
        city,
        forecast_date,
        weather_main,
        COUNT(*) AS frequency
    FROM daily_weather
    WHERE weather_main IS NOT NULL
    GROUP BY city, forecast_date, weather_main
),

most_frequent_weather AS (
    SELECT
        city,
        forecast_date,
        weather_main
    FROM weather_frequency
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY city, forecast_date
        ORDER BY frequency DESC, weather_main ASC
    ) = 1
)

SELECT
    metrics.*,
    weather.weather_main AS most_frequent_weather_main
FROM daily_metrics AS metrics
LEFT JOIN most_frequent_weather AS weather
    ON metrics.city = weather.city
   AND metrics.forecast_date = weather.forecast_date;